In [1]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.normalize as N, src.keys as KY, src.blocking as B, src.rerank as R, src.features as F, src.metrics as M

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
VALID_GT = ROOT / "notebooks" / "phase2_valid_gt.parquet"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=8, memory_limit="5GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)                      # tl() is now used by blocking keys

S1_ALL    = k2("train_source1")
TGT_PARTS = [k2("train_source2"), k2("train_source3")]
TGT       = "(" + " UNION ALL ".join(f"SELECT * FROM {p}" for p in TGT_PARTS) + ")"
ALL_SRC   = [f"{s}_source{i}" for s in ("train", "test") for i in (1, 2, 3)]
print(sys.executable)

D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Scripts\python.exe


In [3]:
importlib.reload(N)

<module 'src.normalize' from 'D:\\ML Challenge\\6ab10eb3b23ba_student_resource\\student_resource\\src\\normalize.py'>

In [4]:
for s in ["Lotus Care Care", "Lotus Care Private Limited", "Zephon Hotels LLC", "Zephon Hotel LLC",
          "Private Smart Exports Limited", "Smart Exports Private Limited",
          "Physical Partners Therapy", "Physical Therapy Partners", "Regional Investments Investments Networks",
          "Maison Comice et Fils", "SARL Maison Comice", "Sri Shiv Solutions Pvt Ltd", "Business Services"]:
    v = N.normalize_name_v2(s)
    print(f"{s!r:44} -> {v!r:42} core={N.name_core(v)!r}")
sql = B.keys_sql("x", "y", "us")
print("new keys present:", all(t in sql for t in ("'srt'", "'p8'", "'s8'", "'p6'")))

'Lotus Care Care'                            -> 'lotus care care'                          core='lotus care'
'Lotus Care Private Limited'                 -> 'lotus care private limited'               core='lotus care'
'Zephon Hotels LLC'                          -> 'zephon hotels llc'                        core='zephon hotel'
'Zephon Hotel LLC'                           -> 'zephon hotel llc'                         core='zephon hotel'
'Private Smart Exports Limited'              -> 'private smart exports limited'            core='smart export'
'Smart Exports Private Limited'              -> 'smart exports private limited'            core='smart export'
'Physical Partners Therapy'                  -> 'physical partners therapy'                core='physical partner therapy'
'Physical Therapy Partners'                  -> 'physical therapy partners'                core='physical therapy partner'
'Regional Investments Investments Networks'  -> 'regional investments investments networks' 

In [5]:
t0 = time.time()
KY.build_all(con, PARQ, KEYS2, ALL_SRC, overwrite=True)
print(f"keys rebuilt in {(time.time() - t0) / 60:.1f} min")

B.run_blocking(con, "cand2", S1_ALL, TGT_PARTS, s1_ids_tbl="eval_s1", k_max=50)
display(pd.concat([B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"cycle3_top{k}/src")
                   for k in (5, 10, 20, 50)], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source1      126.7s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   keys train_source2      271.3s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand2 us       target keys      191s
   cand2 us       key weights      191s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,cycle3_top5/src,0.949879,0.996725,0.985183,9.97265,10.0,10.0,10
1,cycle3_top10/src,0.966283,0.997359,0.989388,19.84595,20.0,20.0,20
2,cycle3_top20/src,0.973012,0.997940,0.991501,39.19665,40.0,40.0,40
3,cycle3_top50/src,0.978990,0.998362,0.993324,93.49275,100.0,100.0,100


In [6]:
display(con.execute(f"""
WITH m AS (SELECT * FROM miss_c),
s AS (SELECT eid(entity_id) AS id, country_key AS c, name_core, replace(name_core, ' ', '') AS cmp FROM {S1_ALL}),
t AS (SELECT eid(entity_id) AS id, country_key AS c, name_core, replace(name_core, ' ', '') AS cmp FROM {TGT}),
j AS (SELECT m.s1_id, m.tgt_id, s.c, s.name_core AS s_core, left(s.cmp, 8) AS s_p8,
             s.name_core = t.name_core                                                     AS core_eq,
             s.cmp = t.cmp                                                                 AS cmp_eq,
             array_to_string(list_sort(tl(s.name_core)), ' ')
               = array_to_string(list_sort(tl(t.name_core)), ' ')                          AS srt_eq,
             length(s.cmp) >= 10 AND length(t.cmp) >= 10 AND left(s.cmp, 8)  = left(t.cmp, 8)  AS p8_eq,
             length(s.cmp) >= 10 AND length(t.cmp) >= 10 AND right(s.cmp, 8) = right(t.cmp, 8) AS s8_eq
      FROM m JOIN s ON s.id = m.s1_id JOIN t ON t.id = m.tgt_id),
core_sz AS (SELECT c, name_core, count(*) AS n FROM t GROUP BY 1, 2),
p8_sz   AS (SELECT c, left(cmp, 8) AS p8, count(*) AS n FROM t WHERE length(cmp) >= 10 GROUP BY 1, 2),
now     AS (SELECT s1_id, tgt_id, min(rk) AS rk FROM cand2 GROUP BY 1, 2)
SELECT count(*)                                                         AS n_missed_cycle2,
       avg(core_eq::INT) AS core, avg(cmp_eq::INT) AS cmp, avg(srt_eq::INT) AS srt,
       avg(p8_eq::INT)   AS p8,   avg(s8_eq::INT)  AS s8,
       avg((core_eq OR cmp_eq OR srt_eq OR p8_eq OR s8_eq)::INT)        AS any_simple_key,
       avg(CASE WHEN core_eq THEN (cs.n > 300)::INT END)                AS core_block_over_cap,
       avg(CASE WHEN p8_eq   THEN (ps.n > 300)::INT END)                AS p8_block_over_cap,
       avg((now.rk IS NOT NULL)::INT)                                   AS now_in_top50,
       avg(CASE WHEN core_eq OR cmp_eq OR srt_eq OR p8_eq OR s8_eq
                THEN (now.rk IS NOT NULL)::INT END)                     AS keyed_and_now_in_top50
FROM j
LEFT JOIN core_sz cs ON cs.c = j.c AND cs.name_core = j.s_core
LEFT JOIN p8_sz   ps ON ps.c = j.c AND ps.p8 = j.s_p8
LEFT JOIN now USING (s1_id, tgt_id)
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_missed_cycle2,core,cmp,srt,p8,s8,any_simple_key,core_block_over_cap,p8_block_over_cap,now_in_top50,keyed_and_now_in_top50
0,1622,0.205302,0.228113,0.211467,0.472873,0.26942,0.553637,0.393393,0.675359,0.199137,0.219376


In [7]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

B.run_blocking(con, "cand_tr", S1_ALL, TGT_PARTS, s1_ids_tbl="train_s1", k_max=50)

tr_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({R.feat_sql('cand_tr', S1_ALL, TGT)}) f LEFT JOIN train_pairs p USING (s1_id, tgt_id)""").fetchdf()
ev_df = con.execute(f"""SELECT f.*, (p.tgt_id IS NOT NULL)::INT AS y
    FROM ({R.feat_sql('cand2', S1_ALL, TGT)}) f LEFT JOIN eval_pairs p USING (s1_id, tgt_id)""").fetchdf()

rr = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
rr.fit(tr_df[R.FEATS], tr_df["y"])
rr.booster_.save_model(str(ARTIFACTS / "rerank_lgb_c3.txt"))
ev_df["p"] = rr.predict_proba(ev_df[R.FEATS])[:, 1]
print(f"eval AUC: {roc_auc_score(ev_df.y, ev_df.p):.4f}")

con.register("ev_scored", ev_df[["s1_id", "tgt_id", "src", "p"]])
con.execute("""CREATE OR REPLACE TABLE cand_rr AS
    SELECT *, row_number() OVER (PARTITION BY s1_id, src ORDER BY p DESC, tgt_id) AS rk2 FROM ev_scored""")
display(pd.concat([B.eval_blocking(con, f"(SELECT s1_id, tgt_id FROM cand_rr WHERE rk2 <= {cap} AND p >= {tau})",
                                   f"p>={tau} cap{cap}")
                   for tau in (0.001, 0.003) for cap in (8, 10)], ignore_index=True))
del tr_df, ev_df; gc.collect()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   cand_tr india    token df           6s
   cand_tr india    S1 keys            7s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

eval AUC: 0.9989


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,p>=0.001 cap8,0.977286,0.998362,0.992927,7.75785,7.0,16.0,16
1,p>=0.001 cap10,0.977459,0.998362,0.992959,7.94760,7.0,20.0,20
2,p>=0.003 cap8,0.976347,0.998362,0.992703,6.64810,6.0,16.0,16
3,p>=0.003 cap10,0.976463,0.998362,0.992726,6.73310,6.0,18.0,20


77

In [8]:
RAW_S1 = pq("train_source1")
RAW_T  = f"(SELECT * FROM {pq('train_source2')} UNION ALL SELECT * FROM {pq('train_source3')})"

con.execute("""
CREATE OR REPLACE TABLE audit_pairs AS
WITH ev AS (SELECT r.s1_id, r.tgt_id, r.p FROM cut_rev r SEMI JOIN eval_s1 USING (s1_id))
SELECT ev.*,
       CASE WHEN g.s1_id = ev.s1_id  THEN 'true match'
            WHEN g.tgt_id IS NULL    THEN 'unowned (distractor)'
            ELSE                          'owned by another S1' END AS kind
FROM ev LEFT JOIN gt_all g ON g.tgt_id = ev.tgt_id
WHERE ev.p >= 0.5 OR g.s1_id = ev.s1_id
""")

base = f"""
FROM audit_pairs a
JOIN (SELECT eid(entity_id) AS id, business_name, business_address FROM {RAW_S1}) s ON s.id = a.s1_id
JOIN (SELECT eid(entity_id) AS id, business_name, business_address FROM {RAW_T})  t ON t.id = a.tgt_id
JOIN (SELECT eid(entity_id) AS id, name_core, addr_v2 FROM {S1_ALL}) ks ON ks.id = a.s1_id
JOIN (SELECT eid(entity_id) AS id, name_core, addr_v2 FROM {TGT})    kt ON kt.id = a.tgt_id"""

display(con.execute(f"""
SELECT kind, count(*) AS n,
       avg((s.business_name = t.business_name)::INT)                             AS raw_name_identical,
       avg((lower(s.business_name) = lower(t.business_name))::INT)               AS raw_name_eq_ignorecase,
       avg((coalesce(s.business_address, '#') = coalesce(t.business_address, '')) ::INT) AS raw_addr_identical,
       avg((t.business_address IS NULL)::INT)                                    AS tgt_addr_missing,
       avg((ks.name_core = kt.name_core)::INT)                                   AS core_name_eq,
       avg((coalesce(ks.addr_v2, '#') = coalesce(kt.addr_v2, ''))::INT)          AS norm_addr_eq,
       avg((t.business_name <> lower(t.business_name) AND t.business_name <> upper(t.business_name))::INT) AS tgt_mixed_case,
       avg((t.business_name = upper(t.business_name))::INT)                      AS tgt_all_caps,
       avg(regexp_matches(t.business_name, '[()\\[\\]#*]')::INT)                 AS tgt_has_brackets_symbols
{base} GROUP BY kind ORDER BY kind""").fetchdf().T)

# H2: how does the house number differ?
display(con.execute(f"""
WITH d AS (SELECT kind,
                  abs(TRY_CAST(regexp_extract(ks.addr_v2, '\\d+') AS BIGINT)
                    - TRY_CAST(regexp_extract(kt.addr_v2, '\\d+') AS BIGINT)) AS dnum,
                  regexp_extract(ks.addr_v2, '\\d+') AS n1, regexp_extract(kt.addr_v2, '\\d+') AS n2
           {base})
SELECT kind, count(*) AS n,
       avg((dnum IS NULL)::INT)                          AS no_number_pair,
       avg((dnum = 0)::INT)                              AS same_number,
       avg((dnum BETWEEN 1 AND 9)::INT)                  AS diff_1_9,
       avg((dnum BETWEEN 10 AND 99)::INT)                AS diff_10_99,
       avg((dnum >= 100 AND (starts_with(n1, n2) OR starts_with(n2, n1)
                             OR ends_with(n1, n2) OR ends_with(n2, n1)))::INT) AS truncated,
       avg((dnum >= 100 AND NOT (starts_with(n1, n2) OR starts_with(n2, n1)
                                 OR ends_with(n1, n2) OR ends_with(n2, n1)))::INT) AS diff_100_plus
FROM d GROUP BY kind ORDER BY kind""").fetchdf().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0,1,2
kind,owned by another S1,true match,unowned (distractor)
n,1159,67630,2353
raw_name_identical,0.018982,0.046355,0.000425
raw_name_eq_ignorecase,0.040552,0.109212,0.0017
raw_addr_identical,0.019845,0.022978,0.0
tgt_addr_missing,0.332183,0.032441,0.00765
core_name_eq,0.402934,0.619311,0.60561
norm_addr_eq,0.086281,0.233609,0.043349
tgt_mixed_case,0.765315,0.748736,0.835954
tgt_all_caps,0.179465,0.182478,0.133447


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0,1,2
kind,owned by another S1,true match,unowned (distractor)
n,1159,67630,2353
no_number_pair,0.438309,0.121987,0.051424
same_number,0.637481,0.835416,0.280914
diff_1_9,0.030722,0.019788,0.360215
diff_10_99,0.082949,0.026423,0.235663
truncated,0.019969,0.044409,0.018817
diff_100_plus,0.192506,0.067119,0.099786


In [9]:
# matches per S1, per source
display(con.execute(f"""
SELECT n_s2, n_s3, count(*) AS n_s1
FROM (SELECT g.source1_entity_id,
             len(list_filter(string_split(coalesce(g.matched_entity_ids, ''), ','), x -> trim(x) LIKE 'S2-%')) AS n_s2,
             len(list_filter(string_split(coalesce(g.matched_entity_ids, ''), ','), x -> trim(x) LIKE 'S3-%')) AS n_s3
      FROM {pq('train_ground_truth')} g)
GROUP BY ALL ORDER BY n_s1 DESC LIMIT 15""").fetchdf())

# unowned targets: how many look like a near-copy of some S1 (claimed with p >= 0.5)?
display(con.execute(f"""
WITH un AS (SELECT eid(entity_id) AS id FROM {TGT} ANTI JOIN gt_all g ON g.tgt_id = eid(entity_id)),
     cl AS (SELECT DISTINCT tgt_id FROM cut_rev WHERE p >= 0.5)
SELECT count(*) AS unowned_targets,
       avg((cl.tgt_id IS NOT NULL)::INT) AS claimed_with_p_ge_0_5
FROM un LEFT JOIN cl ON cl.tgt_id = un.id""").fetchdf())

,n_s2,n_s3,n_s1
0,1,1,269681
1,1,2,251224
2,2,1,223054
3,2,2,208159
4,1,3,140393
5,0,0,123247
6,2,3,116076
7,3,1,114405
8,3,2,105817
9,0,1,60407


,unowned_targets,claimed_with_p_ge_0_5
0,2681854,0.095263


In [10]:
display(con.execute(f"""
WITH s AS (SELECT entity_id, file_row_number AS rs
           FROM read_parquet('{(PARQ / 'train_source1.parquet').as_posix()}', file_row_number=true)),
     t AS (SELECT entity_id, file_row_number AS rt
           FROM read_parquet('{(PARQ / 'train_source2.parquet').as_posix()}', file_row_number=true)),
     g AS (SELECT s1_id, tgt_id FROM gt_all WHERE tgt_id // {B.SRC} = 2)
SELECT corr(s.rs, t.rt)                                        AS row_order_corr,
       corr(g.s1_id % {B.SRC}, g.tgt_id % {B.SRC})             AS id_number_corr,
       avg((abs(s.rs / 2206821.0 - t.rt / 5034616.0) < 0.001)::INT) AS same_relative_position
FROM g JOIN s ON eid(s.entity_id) = g.s1_id JOIN t ON eid(t.entity_id) = g.tgt_id
""").fetchdf())

,row_order_corr,id_number_corr,same_relative_position
0,0.001246,0.000117,0.002004


In [12]:
con.execute(f"""
CREATE OR REPLACE TABLE audit_txt AS
WITH ids_s AS (SELECT DISTINCT s1_id  AS id FROM audit_pairs),
     ids_t AS (SELECT DISTINCT tgt_id AS id FROM audit_pairs),
     rs AS (SELECT eid(entity_id) AS id, business_name, business_address FROM {RAW_S1}
            WHERE eid(entity_id) IN (SELECT id FROM ids_s)),
     rt AS (SELECT eid(entity_id) AS id, business_name, business_address FROM {RAW_T}
            WHERE eid(entity_id) IN (SELECT id FROM ids_t)),
     ks AS (SELECT eid(entity_id) AS id, addr_v2 FROM {S1_ALL} WHERE eid(entity_id) IN (SELECT id FROM ids_s)),
     kt AS (SELECT eid(entity_id) AS id, addr_v2 FROM {TGT}    WHERE eid(entity_id) IN (SELECT id FROM ids_t))
SELECT a.*, rs.business_name AS s_raw_name, rs.business_address AS s_raw_addr,
       rt.business_name AS t_raw_name, rt.business_address AS t_raw_addr,
       ks.addr_v2 AS s_addr, kt.addr_v2 AS t_addr,
       regexp_extract(coalesce(ks.addr_v2, ''), '\\d+') AS s_n1,
       regexp_extract(coalesce(kt.addr_v2, ''), '\\d+') AS t_n1
FROM audit_pairs a
JOIN rs ON rs.id = a.s1_id JOIN rt ON rt.id = a.tgt_id
JOIN ks ON ks.id = a.s1_id JOIN kt ON kt.id = a.tgt_id
""")
print(con.execute("SELECT kind, count(*) FROM audit_txt GROUP BY 1").fetchall())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('owned by another S1', 1159), ('true match', 67630), ('unowned (distractor)', 2353)]


In [13]:
pd.set_option("display.max_colwidth", 45)
print("=== DISTRACTORS with the SAME house number ===")
display(con.execute("""
SELECT round(p, 3) AS p, s_raw_name, t_raw_name, s_raw_addr, t_raw_addr
FROM audit_txt
WHERE kind = 'unowned (distractor)' AND s_n1 <> '' AND s_n1 = t_n1
ORDER BY p DESC LIMIT 30""").fetchdf())

print("=== TRUE MATCHES with the SAME house number (contrast) ===")
display(con.execute("""
SELECT round(p, 3) AS p, s_raw_name, t_raw_name, s_raw_addr, t_raw_addr
FROM audit_txt
WHERE kind = 'true match' AND s_n1 <> '' AND s_n1 = t_n1
ORDER BY hash(s1_id + tgt_id) LIMIT 15""").fetchdf())

=== DISTRACTORS with the SAME house number ===


,p,s_raw_name,t_raw_name,s_raw_addr,t_raw_addr
0,1.000,Saam Brothers Private Limited,Svojas &-Brothers [Exports],"House No. 421, Neel Kanth Appartments, Se...","421/15, Xxi1864a, East Chalakudy, Thrissu..."
1,1.000,Diamond Elite Home Care LLC,VC Diamond Bain ínc,"1300 Rimtop Drive, Billings, MT","1300, Lehi, Utah"
2,1.000,Hancox and Wells LLC,"Bradley, Vivyanne, CPA LLC","Chester, NY, 142 Main Street","142 Main Street, New Palestine, Indiana"
3,1.000,Electronic Software Studios,Desert Best Starry East Care,"131 Bittinger Court, Martinsburg, WV","131 DEWEY DRIVE, MASVILLE CITY, WV"
4,1.000,"Thomas, Class & Tweed Marketing LLC",Thomas Research Corp,"411 Woodland Drive, Town Of Beloit, WI","411 MONROE ST, COLONIAL BEACH TOWN, VA"
5,1.000,Ekvira Consultant Ltd,Ekvira Consultant Pvt Ltd,"Plot No-04, Allapur Village Borabanda, Hy...","Plot No-4, Allapur Village Borabanda, Hyd..."
6,1.000,"Peters, Annmaria E., Ph.D.","Peters, Annmaria E., Ph.D. Inc","2170 Sherrills Ford Road, Salisbury, NC","2170 SHERRILLS FORD ROAD, SALISBURY, NC"
7,1.000,My Power Private Limited,Alpha Infotech Private Limited,"H No 13, Delhi, West Delhi, 1St Floor Roa...","13/54 WEST PUNJABI BAGH, DELHI, Delhi"
8,1.000,Regional Investment Networks P.C.,Regional Investment Investment Networks Llc,"6695 St Rte 15, Unit 22, Bryan, OH","6695 St Rte 15, Unit 22, Bryan, Ohio"
9,1.000,"Eastern Heritage Surgical, Inc","HERITAGE AGRICULTURAL SERVICES, PARTNERS CO","236 Washington Street, Shelbyville, IN","236 DIAGONAL ST, GALEAN CITY, IL"


=== TRUE MATCHES with the SAME house number (contrast) ===


,p,s_raw_name,t_raw_name,s_raw_addr,t_raw_addr
0,1.000,Sai Software Private Limited,சாய் சாஃப்ட்வேர் பிரைவேட் லிமிடெட்,"Khcentre, No.15/2, Collegeroadnungambakka...","KHCENTRE, NO.15/2, COLLEGEROADNUNGAMBAKKA..."
1,1.000,Surgical Green Specialists,The Surgical Green Specialists,"12980 Bowdoin Drive, TX, El Paso County","0012980 BOWDOIN DRIVE, EL PASO COUNTY, TX"
2,0.987,Oconnor & Sharma LLC,Oconnor & 5hara LLC,"1705 Shelby Street, Salem, IN","1705 SHELBY ST, SALEM, IN"
3,0.978,Environmental Institute P.C.,Environmental Issiigute P.C.,"Belgrade, MT, 110 7th Street","110 7nd St, Belgrade, Montana"
4,1.000,Smart Freight Enterprises LLC,The Smart Freight Enterprises [LLC],"4123 200, Greensburg, IN","4123 200, GREENSBURG, IN"
5,0.999,Parel Service Private Limited,Parel Service Private-Limited,"Flat No. 61, Wing - A, Dr. S.S.Road, Kalp...","61 , WING - A, DR. S.S.ROAD, KALPATRU HAB..."
6,0.998,Barrow Strategic Graf Inc,Barrow Strategic Graf Inc,"213 Boulder Road, Solvay, NY","Syracuse, 213 Boulder Road, New York"
7,0.997,Sonu Plast Private Limited,Sonu Plast Private (Límited),"No.22/13, Sanjay Gandhi St Mgr Nagar, Che...","No.22/13, TN, K K Nagar"
8,1.000,Montgomery Superior Surgical LLC,Montgomery Superior Surgical LLC,"MA, 48 Taber Avenue, Brockton, Unit 2","48 Taber Ave, Unit 2, Brockton, Massachus..."
9,0.907,Riify,Riify Center,"129 6th Street, Oswego, NY","129-131 6th Street, Oswego, New York"


In [14]:
importlib.reload(B); importlib.reload(R); importlib.reload(F)
con.execute("PRAGMA threads=8"); con.execute("SET memory_limit='5GB'")
F.setup_macros(con)

# 1. materialize keys as native DuckDB tables (read once, scanned many times)
t0 = time.time()
con.execute(f"CREATE OR REPLACE TABLE k_s1 AS SELECT * FROM {k2('train_source1')}")
con.execute(f"CREATE OR REPLACE TABLE k_s2 AS SELECT * FROM {k2('train_source2')}")
con.execute(f"CREATE OR REPLACE TABLE k_s3 AS SELECT * FROM {k2('train_source3')}")
print(f"materialized in {time.time() - t0:.0f}s")

S1_T      = "k_s1"
TGT_PARTS_T = ["k_s2", "k_s3"]
TGT_T     = "(SELECT * FROM k_s2 UNION ALL SELECT * FROM k_s3)"

# 2. full blocking + rerank + cut, 32 S1 chunks
TAU_CUT, CAP_CUT, K_MAX = 0.003, 8, 50
reranker = R.load(ARTIFACTS / "rerank_lgb_c3.txt")

con.execute("CREATE OR REPLACE TABLE cut_full (s1_id BIGINT, tgt_id BIGINT, src BIGINT, p DOUBLE, rk2 BIGINT)")
t0 = time.time()
B.run_blocking(con, "cut_full", S1_T, TGT_PARTS_T, s1_ids_tbl=None, k_max=K_MAX, s1_chunks=32,
               on_chunk=lambda tbl: R.rerank_and_cut(con, tbl, S1_T, TGT_T, reranker, "cut_full",
                                                     tau=TAU_CUT, cap=CAP_CUT))
print(f"done in {(time.time() - t0) / 60:.1f} min")
print(con.execute("SELECT count(*), count(DISTINCT s1_id), count(*) / count(DISTINCT s1_id) FROM cut_full").fetchone())
display(B.eval_blocking(con, "(SELECT s1_id, tgt_id FROM cut_full)", "cycle3 full cut, eval S1"))

F.build_rev(con, "cut_full", "cut_rev")
F.build_name_idf(con, TGT_T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

materialized in 25s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

   cut_full us       scored 32/32   10970s
done in 182.9 min
(14676094, 2206473, 6.651381639385572)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,"cycle3 full cut, eval S1",0.976347,0.998521,0.992832,6.65115,6.0,16.0,16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [1]:
import sys, importlib
from pathlib import Path
from collections import Counter
import duckdb, pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.normalize as N, src.blocking as B

ARTIFACTS = ROOT / "artifacts"
con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=2, memory_limit="2GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
print(con.execute("SELECT kind, count(*) FROM audit_txt GROUP BY 1").fetchall())

[('true match', 67630), ('unowned (distractor)', 2353), ('owned by another S1', 1159)]


In [2]:
df = con.execute("SELECT kind, s_raw_name, t_raw_name FROM audit_txt").fetchdf()
LEG = N.LEGAL_FORMS - {"and"}
nm = {x: (N.normalize_name_v2(x) or "") for x in set(df.s_raw_name) | set(df.t_raw_name)}

def legal_cat(a, b):
    sa = tuple(t for t in nm[a].split() if t in LEG)
    tb = tuple(t for t in nm[b].split() if t in LEG)
    if not sa and not tb: return "both none"
    if sa == tb:          return "same"
    if sa and not tb:     return "dropped in target"
    if tb and not sa:     return "added in target"
    if set(sa) < set(tb) or set(tb) < set(sa): return "subset/superset"
    return "changed"

df["legal"] = [legal_cat(a, b) for a, b in zip(df.s_raw_name, df.t_raw_name)]
print("=== legal-form transition, share within each kind ===")
display(pd.crosstab(df.legal, df.kind, normalize="columns").round(3))

cnt = {k: Counter() for k in df.kind.unique()}
for k, a, b in zip(df.kind, df.s_raw_name, df.t_raw_name):
    for t in set(nm[b].split()) - set(nm[a].split()):
        cnt[k][t] += 1
tm, dx = cnt["true match"], cnt["unowned (distractor)"]
ntm, ndx = (df.kind == "true match").sum(), (df.kind == "unowned (distractor)").sum()
tok = pd.DataFrame([(t, tm[t] / ntm, dx[t] / ndx) for t in set(tm) | set(dx) if tm[t] + dx[t] >= 15],
                   columns=["added_token", "rate_true", "rate_distractor"])
tok["distractor_ratio"] = (tok.rate_distractor + 1e-4) / (tok.rate_true + 1e-4)
print("=== added tokens most typical of DISTRACTORS ===")
display(tok.sort_values("distractor_ratio", ascending=False).head(20).round(4))
print("=== added tokens most typical of TRUE MATCHES ===")
display(tok.sort_values("distractor_ratio").head(15).round(4))

=== legal-form transition, share within each kind ===


kind,owned by another S1,true match,unowned (distractor)
legal,,,
added in target,0.135,0.062,0.271
both none,0.155,0.264,0.070
changed,0.155,0.014,0.326
dropped in target,0.248,0.151,0.041
same,0.219,0.462,0.170
subset/superset,0.088,0.046,0.122


=== added tokens most typical of DISTRACTORS ===


,added_token,rate_true,rate_distractor,distractor_ratio
25,private,0.0000,0.0612,612.9847
10,holdings,0.0000,0.0110,111.4972
37,infratech,0.0000,0.0089,90.2478
4,enterprises,0.0000,0.0102,89.7297
53,overseas,0.0000,0.0085,85.9979
9,industries,0.0000,0.0081,81.7480
19,group,0.0000,0.0076,77.4981
57,public,0.0000,0.0072,73.2482
46,exports,0.0000,0.0068,68.9983
16,ventures,0.0000,0.0064,64.7484


=== added tokens most typical of TRUE MATCHES ===


,added_token,rate_true,rate_distractor,distractor_ratio
30,dba,0.0056,0.0,0.0176
42,formerly,0.0040,0.0,0.0245
38,as,0.0037,0.0,0.0260
60,f/k/a,0.0020,0.0,0.0470
49,aka,0.0015,0.0,0.0622
45,t/a,0.0014,0.0,0.0685
41,a/k/a,0.0013,0.0,0.0721
18,doing,0.0013,0.0,0.0721
11,d/b/a,0.0012,0.0,0.0745
35,known,0.0010,0.0,0.0929


In [5]:
import importlib, src.features as F
importlib.reload(F)
print(len(F.FEATS_V4))                                    # 92
print(F._alias_parts("acme holdings llc dba sunrise bakery"))  # ['acme holdings llc', 'sunrise bakery']

85
['acme holdings llc', 'sunrise bakery']


In [6]:
print(F.FEATS_V4[-4:])   # ['t_has_alias', 's_has_alias', 'alias_best_tset', 'alias_best_jw']

['t_has_alias', 's_has_alias', 'alias_best_tset', 'alias_best_jw']


In [2]:
PARQ = ARTIFACTS / "parquet"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"

In [3]:
from rapidfuzz.distance import JaroWinkler

df = con.execute(f"""
SELECT a.s1_id, a.tgt_id, round(a.p, 3) AS p,
       a.s_raw_name AS cand_name,  a.s_raw_addr AS cand_addr,
       o.business_name AS owner_name, o.business_address AS owner_addr,
       a.t_raw_name AS tgt_name,  a.t_raw_addr AS tgt_addr
FROM audit_txt a
JOIN gt_all g ON g.tgt_id = a.tgt_id
JOIN (SELECT eid(entity_id) AS id, business_name, business_address FROM {pq('train_source1')}) o ON o.id = g.s1_id
WHERE a.kind = 'owned by another S1'
""").fetchdf()

LEG = N.LEGAL_FORMS - {"and"}
def nv(x):   return N.normalize_name_v2(x) or ""
def core(x): return N.name_core(nv(x)) or ""
def lf(x):   return tuple(t for t in nv(x).split() if t in LEG)
def atoks(x): return set((N.normalize_address_v2(x) or "").split())
def jac(a, b): return len(a & b) / len(a | b) if a and b else 0.0

rows = []
for r in df.itertuples():
    tn, cn, on = core(r.tgt_name), core(r.cand_name), core(r.owner_name)
    ta, ca, oa = atoks(r.tgt_addr), atoks(r.cand_addr), atoks(r.owner_addr)
    rows.append({
        "cand_owner_same_core": cn == on,
        "tgt_addr_missing":     not ta,
        "name: owner closer":   JaroWinkler.normalized_similarity(tn, on) > JaroWinkler.normalized_similarity(tn, cn) + 0.01,
        "name: cand closer":    JaroWinkler.normalized_similarity(tn, cn) > JaroWinkler.normalized_similarity(tn, on) + 0.01,
        "legal: owner agrees only": (lf(r.tgt_name) == lf(r.owner_name)) and (lf(r.tgt_name) != lf(r.cand_name)),
        "legal: cand agrees only":  (lf(r.tgt_name) == lf(r.cand_name)) and (lf(r.tgt_name) != lf(r.owner_name)),
        "addr: owner closer":   bool(ta) and jac(ta, oa) > jac(ta, ca) + 0.05,
        "addr: cand closer":    bool(ta) and jac(ta, ca) > jac(ta, oa) + 0.05,
    })
res = pd.DataFrame(rows)
res["any_signal_for_owner"] = res["name: owner closer"] | res["legal: owner agrees only"] | res["addr: owner closer"]
res["no_signal_at_all"] = ~(res["any_signal_for_owner"] | res["name: cand closer"]
                            | res["legal: cand agrees only"] | res["addr: cand closer"])
print(f"contested pairs: {len(res)}")
display(res.mean().round(3).to_frame("share").T)

pd.set_option("display.max_colwidth", 38)
display(df[["p", "cand_name", "owner_name", "tgt_name", "cand_addr", "owner_addr", "tgt_addr"]]
        .sample(20, random_state=1))

contested pairs: 1159


,cand_owner_same_core,tgt_addr_missing,name: owner closer,name: cand closer,legal: owner agrees only,legal: cand agrees only,addr: owner closer,addr: cand closer,any_signal_for_owner,no_signal_at_all
share,0.404,0.332,0.544,0.049,0.374,0.059,0.425,0.011,0.796,0.174


,p,cand_name,owner_name,tgt_name,cand_addr,owner_addr,tgt_addr
830,0.616,Pediatric Dental Center Care,Pediatric Dental Center Care,Pediatric Dental Center [Ce],"707 Washington Street, Monroe, IA","6711 Thornwood Street, Canton, OH",NaN
202,0.619,Bell (India) Homes Ltd,Galaxy Technologies Private Limited,Galaxy Technologies Private Ltd,"556, New Pargaon, Hatkanangale, Sh...","556, New Pargaon, Hatkanangale, Sh...","556, New Pargaon, Hatkanangale, Sh..."
972,0.584,Super Infotech Private Limited,Super Infotech Pvt Ltd,सुपर इंफोटेक प्रा. लि.,"E-208, Sector-63, Noida, Gautam Bu...","F-398, Sector 63, Gautam Budh Naga...","NOIDA, SECTOR 63, GAUTAM BUDH NAGA..."
762,1.000,Memphis Family Frontier Care,Brana Laird Dominion,Brana Láird Dominion,"1506 Morehead Street, Memphis, TN","1506 47th Street, Chattanooga, TN","1506 47rd St, Chattanooga, Tennessee"
437,0.787,Heartland Technology,Primary Care Clinic,@Primarycare #77757,"2066 Faye Street, Mobile, AL","2066 Faye Street, Mobile, AL","2066 Faye St, Mobie Township, Alabama"
930,0.693,Valency Energy (India) Private Lim...,Devya Buildersprivate Pvt Ltd,Devya Buildersprivate Pvt (Limited),"Plot 183, 185 / 192 Part, 3Rd Main...","Tamil Nadu, Plot 183, 185 / 192 Pa...","PLOT 183, 185 / 192 PART, 3RD MAIN..."
553,0.654,Rosemary Services Private Limited,Rosemary Motors Ltd,Rosemary Ltd [Services],"C/228, Indraloknagar, C H S Ltd, N...","Swapnalok -Ii, Marve Road, Malad W...",NaN
81,0.769,Vemisoft LLC,Vemisoft Co,Vemisoft (Co),"110 Murrow Lane, Jacksonville, NC","1013 46 Street, Brooklyn, NY",NaN
220,0.514,Olanara Street Care,Olanara Street Care,Olanara Street Care,"14 Gaewood Avenue, Wheeling, WV","12111 104th Avenue, Unit #D402, Pu...",NaN
772,0.582,Delta Dental Associates,Vebriva,Vebriva LP,"16845 Oster Point Road, Cold Sprin...","16207 Oster Point Road, Cold Sprin...","Cold Spring, Minnesota, Oster Poin..."


In [1]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M
ARTIFACTS = ROOT / "artifacts"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="4GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)
S1_ALL, TGT = "k_s1", "(SELECT * FROM k_s2 UNION ALL SELECT * FROM k_s3)"

# placeholders so the v4 SQL runs (v3 ignores these columns)
con.execute("CREATE TABLE IF NOT EXISTS tok_odds (kind VARCHAR, tok VARCHAR, lo DOUBLE, n BIGINT)")
con.execute("CREATE TABLE IF NOT EXISTS addr_idf (c VARCHAR, tok VARCHAR, idf DOUBLE)")
con.execute("CREATE TABLE IF NOT EXISTS raw_names (id BIGINT, raw VARCHAR)")

con.execute("CREATE OR REPLACE TABLE q_tr_s1 AS SELECT s1_id FROM m_tr_s1 USING SAMPLE 300000 ROWS (reservoir, 3)")
con.execute("""CREATE OR REPLACE TABLE q_tr_rows AS SELECT r.*, (g.tgt_id IS NOT NULL)::INT AS y
    FROM cut_rev r SEMI JOIN q_tr_s1 USING (s1_id) LEFT JOIN gt_all g USING (s1_id, tgt_id)""")
con.execute("""CREATE OR REPLACE TABLE m_ev_rows AS SELECT r.*, (g.tgt_id IS NOT NULL)::INT AS y
    FROM cut_rev r SEMI JOIN eval_s1 USING (s1_id) LEFT JOIN eval_pairs g USING (s1_id, tgt_id)""")

def fl(rows, n):
    df = F.build_features_chunked(con, rows, S1_ALL, TGT, n_chunks=n, feats=F.FEATS_V3)
    return df.merge(con.execute(f"SELECT s1_id, tgt_id, y FROM {rows}").fetchdf(), on=["s1_id", "tgt_id"])

t0 = time.time()
tr, ev = fl("q_tr_rows", 12), fl("m_ev_rows", 2)
print(f"train {tr.shape} | eval {ev.shape} ({(time.time() - t0) / 60:.1f} min)")

rng = np.random.default_rng(0)
es = tr.s1_id.isin(set(rng.choice(tr.s1_id.unique(), size=tr.s1_id.nunique() // 10, replace=False)))
m = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=4000, learning_rate=0.05, max_depth=10,
                      min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric="logloss", early_stopping_rounds=100)
m.fit(tr.loc[~es, F.FEATS_V3], tr.loc[~es, "y"], eval_set=[(tr.loc[es, F.FEATS_V3], tr.loc[es, "y"])], verbose=False)
m.save_model(str(ARTIFACTS / "matcher_xgb_v3_c3.json"))

ev["q"] = m.predict_proba(ev[F.FEATS_V3])[:, 1]
print(f"AUC {roc_auc_score(ev.y, ev.q):.4f}")
con.register("evq", ev[["s1_id", "tgt_id", "q"]])
display(pd.concat([M.eval_f05(con, f"(SELECT s1_id, tgt_id FROM evq WHERE q >= {t})", f"q>={t}")
                   for t in (0.6, 0.7, 0.8)], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (1995457, 61) | eval (133023, 61) (4.4 min)


D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [21:25:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


AUC 0.9990


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q>=0.6,0.978903,0.966324,0.979613,0.992472,0.957517,3.34065
1,q>=0.7,0.979328,0.973807,0.979640,0.994820,0.953922,3.32025
2,q>=0.8,0.978896,0.979420,0.978866,0.996515,0.949648,3.29975


In [1]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M

ARTIFACTS = ROOT / "artifacts"
con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="3GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)
S1_ALL, TGT = "k_s1", "(SELECT * FROM k_s2 UNION ALL SELECT * FROM k_s3)"

# ---- 1. rebuild eval predictions with the T1 model ----
t0 = time.time()
m = xgb.XGBClassifier()
m.load_model(str(ARTIFACTS / "matcher_xgb_v3_c3.json"))
ev = F.build_features_chunked(con, "m_ev_rows", S1_ALL, TGT, n_chunks=2, feats=F.FEATS_V3)
ev = ev.merge(con.execute("SELECT s1_id, tgt_id, y FROM m_ev_rows").fetchdf(), on=["s1_id", "tgt_id"])
ev["q"] = m.predict_proba(ev[F.FEATS_V3])[:, 1]
print(f"eval {ev.shape} | AUC {roc_auc_score(ev.y, ev.q):.4f} | {time.time() - t0:.0f}s")

# ---- 2. decision-rule search ----
e = ev[["s1_id", "tgt_id", "q"]].copy()
e["rk"]   = e.groupby("s1_id")["q"].rank(ascending=False, method="first")
e["maxq"] = e.groupby("s1_id")["q"].transform("max")

def score(mask, label):
    con.register("dp", e.loc[mask, ["s1_id", "tgt_id"]])
    r = M.eval_f05(con, "(SELECT * FROM dp)", label)
    con.unregister("dp")
    return r

rows = [score(e.q >= 0.7, "global q>=0.7 (current)")]

# A: separate thresholds for an S1's best candidate vs the rest
for t1 in (0.5, 0.6, 0.7, 0.8):
    for t2 in (0.6, 0.7, 0.8, 0.9):
        rows.append(score(((e.rk == 1) & (e.q >= t1)) | ((e.rk > 1) & (e.q >= t2)),
                          f"A top1>={t1} rest>={t2}"))

# B: absolute + relative to the S1's best candidate
for t in (0.6, 0.7):
    for r in (0.3, 0.5, 0.7):
        rows.append(score((e.q >= t) & (e.q >= r * e.maxq), f"B q>={t} & q>={r}*max"))

# C: per-entity expected-F0.5 (plug-in), with probability sharpening a
d = e.sort_values(["s1_id", "q"], ascending=[True, False]).reset_index(drop=True)
for a in (1.0, 1.5, 2.0):
    x = d.copy()
    x["qq"] = x.q.clip(1e-9, 1 - 1e-9) ** a
    g = x.groupby("s1_id", sort=False)
    x["k"] = g.cumcount() + 1
    x["ef"] = 1.25 * g["qq"].cumsum() / (0.25 * g["qq"].transform("sum") + x["k"])
    x["e0"] = np.exp(np.log1p(-x["qq"]).groupby(x["s1_id"]).transform("sum"))
    best = x.loc[x.groupby("s1_id", sort=False)["ef"].idxmax(), ["s1_id", "k", "ef", "e0"]]
    best["kstar"] = np.where(best.ef > best.e0, best.k, 0)
    x = x.merge(best[["s1_id", "kstar"]], on="s1_id")
    keep = x.loc[x.k <= x.kstar, ["s1_id", "tgt_id"]]
    con.register("dp", keep)
    rows.append(M.eval_f05(con, "(SELECT * FROM dp)", f"C expF05 a={a}"))
    con.unregister("dp")

res = pd.concat(rows, ignore_index=True).sort_values("macro_f05", ascending=False)
display(res.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

eval (133023, 62) | AUC 0.9990 | 21s


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
3,A top1>=0.5 rest>=0.8,0.979593,0.958840,0.980765,0.996170,0.950225,3.30290
24,C expF05 a=1.5,0.979577,0.966324,0.980325,0.996528,0.949041,3.29760
7,A top1>=0.6 rest>=0.8,0.979572,0.966324,0.980320,0.996290,0.950081,3.30200
2,A top1>=0.5 rest>=0.7,0.979496,0.958840,0.980662,0.994567,0.954240,3.32220
6,A top1>=0.6 rest>=0.7,0.979475,0.966324,0.980217,0.994686,0.954095,3.32130
19,B q>=0.6 & q>=0.7*max,0.979457,0.966324,0.980199,0.994597,0.954211,3.32200
11,A top1>=0.7 rest>=0.8,0.979426,0.973807,0.979743,0.996425,0.949908,3.30095
23,C expF05 a=1.0,0.979411,0.953227,0.980889,0.995054,0.952796,3.31555
10,A top1>=0.7 rest>=0.7,0.979328,0.973807,0.979640,0.994820,0.953922,3.32025
20,B q>=0.7 & q>=0.3*max,0.979328,0.973807,0.979640,0.994820,0.953922,3.32025


In [2]:
import sys, time, gc
from pathlib import Path
import duckdb, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M
ARTIFACTS = ROOT / "artifacts"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="4GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)
S1_ALL, TGT = "k_s1", "(SELECT * FROM k_s2 UNION ALL SELECT * FROM k_s3)"

def fl(rows, n):
    df = F.build_features_chunked(con, rows, S1_ALL, TGT, n_chunks=n, feats=F.FEATS_V3)
    return df.merge(con.execute(f"SELECT s1_id, tgt_id, y FROM {rows}").fetchdf(), on=["s1_id", "tgt_id"])

# the 300k already used (q_tr_s1) + 300k more from the same pool -> 600k total
con.execute("""CREATE OR REPLACE TABLE q_tr2_s1 AS
    SELECT s1_id FROM (SELECT * FROM m_tr_s1 ANTI JOIN q_tr_s1 USING (s1_id)) USING SAMPLE 300000 ROWS (reservoir, 7)""")
con.execute("""CREATE OR REPLACE TABLE q_tr2_rows AS SELECT r.*, (g.tgt_id IS NOT NULL)::INT AS y
    FROM cut_rev r SEMI JOIN (SELECT s1_id FROM q_tr_s1 UNION ALL SELECT s1_id FROM q_tr2_s1) USING (s1_id)
    LEFT JOIN gt_all g USING (s1_id, tgt_id)""")

t0 = time.time()
tr = fl("q_tr2_rows", 24)
ev = fl("m_ev_rows", 2)
print(f"train {tr.shape} | eval {ev.shape} ({(time.time() - t0) / 60:.1f} min)")

rng = np.random.default_rng(1)
es = tr.s1_id.isin(set(rng.choice(tr.s1_id.unique(), size=tr.s1_id.nunique() // 10, replace=False)))
mb = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=5000, learning_rate=0.04, max_depth=10,
                       min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                       eval_metric="logloss", early_stopping_rounds=120, random_state=1)
mb.fit(tr.loc[~es, F.FEATS_V3], tr.loc[~es, "y"], eval_set=[(tr.loc[es, F.FEATS_V3], tr.loc[es, "y"])], verbose=False)
mb.save_model(str(ARTIFACTS / "matcher_xgb_v3_c3_big.json"))

ms = xgb.XGBClassifier(); ms.load_model(str(ARTIFACTS / "matcher_xgb_v3_c3.json"))
ev["q_small"] = ms.predict_proba(ev[F.FEATS_V3])[:, 1]
ev["q_big"]   = mb.predict_proba(ev[F.FEATS_V3])[:, 1]
ev["q_ens"]   = (ev.q_small + ev.q_big) / 2
print("AUC", {c: round(roc_auc_score(ev.y, ev[c]), 5) for c in ("q_small", "q_big", "q_ens")})

con.register("evq", ev[["s1_id", "tgt_id", "q_small", "q_big", "q_ens"]])
def rule(c):  # the T2 rule: top1 >= 0.7, rest >= 0.8
    return f"""(SELECT s1_id, tgt_id FROM (
        SELECT *, row_number() OVER (PARTITION BY s1_id ORDER BY {c} DESC) AS rk FROM evq WHERE {c} >= 0.7)
        WHERE rk = 1 OR {c} >= 0.8)"""
display(pd.concat([M.eval_f05(con, rule(c), c) for c in ("q_small", "q_big", "q_ens")], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (3990362, 61) | eval (133023, 61) (6.7 min)


D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [21:42:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


AUC {'q_small': 0.99901, 'q_big': 0.99911, 'q_ens': 0.99908}


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q_small,0.979426,0.973807,0.979743,0.996425,0.949908,3.30095
1,q_big,0.980216,0.974743,0.980525,0.996506,0.951279,3.30545
2,q_ens,0.980041,0.975678,0.980288,0.996699,0.950572,3.30235


In [2]:
import sys, time, gc
from pathlib import Path
import duckdb, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M
ARTIFACTS = ROOT / "artifacts"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=14, memory_limit="9GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)
S1_ALL, TGT = "k_s1", "(SELECT * FROM k_s2 UNION ALL SELECT * FROM k_s3)"

def fl(rows, n):
    df = F.build_features_chunked(con, rows, S1_ALL, TGT, n_chunks=n, feats=F.FEATS_V3)
    return df.merge(con.execute(f"SELECT s1_id, tgt_id, y FROM {rows}").fetchdf(), on=["s1_id", "tgt_id"])

t0 = time.time()
tr, ev = fl("q_tr2_rows", 24), fl("m_ev_rows", 2)            # same 600k rows as the big model
print(f"features {(time.time() - t0) / 60:.1f} min")

rng = np.random.default_rng(1)
es = tr.s1_id.isin(set(rng.choice(tr.s1_id.unique(), size=tr.s1_id.nunique() // 10, replace=False)))
X, y, Xv, yv = tr.loc[~es, F.FEATS_V3], tr.loc[~es, "y"], tr.loc[es, F.FEATS_V3], tr.loc[es, "y"]

def fit(name, **kw):
    t = time.time()
    p = dict(tree_method="hist", device="cuda", n_estimators=8000, eval_metric="logloss",
             early_stopping_rounds=150, subsample=0.8, colsample_bytree=0.8)
    p.update(kw)
    mdl = xgb.XGBClassifier(**p); mdl.fit(X, y, eval_set=[(Xv, yv)], verbose=False)
    mdl.save_model(str(ARTIFACTS / f"{name}.json"))
    print(f"{name}: {(time.time() - t) / 60:.1f} min, best iter {mdl.best_iteration}")
    return mdl

mA = fit("matcher_xgb_v3_deep",  max_depth=12, learning_rate=0.03, min_child_weight=3, random_state=2)
mB = fit("matcher_xgb_v3_seed3", max_depth=10, learning_rate=0.03, min_child_weight=5, random_state=3)
mbig = xgb.XGBClassifier(); mbig.load_model(str(ARTIFACTS / "matcher_xgb_v3_c3_big.json"))

for c, mdl in [("q_big", mbig), ("q_deep", mA), ("q_seed3", mB)]:
    ev[c] = mdl.predict_proba(ev[F.FEATS_V3])[:, 1]
ev["q_ens3"] = ev[["q_big", "q_deep", "q_seed3"]].mean(axis=1)
ev["q_ens2"] = ev[["q_big", "q_deep"]].mean(axis=1)

con.register("evq", ev[["s1_id", "tgt_id", "q_big", "q_deep", "q_seed3", "q_ens3", "q_ens2"]])
def rule(c):
    return f"""(SELECT s1_id, tgt_id FROM (
        SELECT *, row_number() OVER (PARTITION BY s1_id ORDER BY {c} DESC) AS rk FROM evq WHERE {c} >= 0.7)
        WHERE rk = 1 OR {c} >= 0.8)"""
res = pd.concat([M.eval_f05(con, rule(c), c) for c in ("q_big", "q_deep", "q_seed3", "q_ens3", "q_ens2")],
                ignore_index=True).sort_values("macro_f05", ascending=False)
display(res)
print(f"total {(time.time() - t0) / 60:.1f} min")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   features 4/24      50s


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features 6.5 min
matcher_xgb_v3_deep: 2.3 min, best iter 1391
matcher_xgb_v3_seed3: 2.3 min, best iter 2545


D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [23:14:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
3,q_ens3,0.980221,0.974743,0.980530,0.996656,0.951092,3.30430
0,q_big,0.980216,0.974743,0.980525,0.996506,0.951279,3.30545
4,q_ens2,0.980191,0.973807,0.980552,0.996551,0.951294,3.30535
2,q_seed3,0.980091,0.974743,0.980393,0.996460,0.951034,3.30475
1,q_deep,0.980077,0.974743,0.980378,0.996476,0.951251,3.30545


total 11.3 min
